# SGDC — dataset retrieval (Mip-NeRF 360)

Downloads the **Mip-NeRF 360** scenes to Google Drive. Preparation (HLOC SfM,
SAM2 labels) is **not** done here — that stays in `Prepare_Datasets.ipynb`, which
reads from `raw/`.

| dataset | source | lands at |
|---|---|---|
| Mip-NeRF 360 | `360_v2.zip` (official, images + COLMAP) | `$RAW_DIR/<scene>` |

Paths follow your existing convention:

```
HOME_DIR     = /content/drive/MyDrive/Magister/Thesis/Script
RAW_DIR      = $HOME_DIR/raw        # Mip-NeRF 360 extraction + scratch
DATASET_ROOT = $HOME_DIR/Dataset    # prepared scenes live here (written by Prepare_Datasets)
```

The zip is downloaded and unzipped on the fast local Colab disk
(`/content/_retrieve`), so only the extracted scene folders land on Drive.

> **Replica is deferred.** §6 keeps the Replica retrieval cell (disabled) for when
you pick a source; it does nothing unless `REPLICA_SCENES` is non-empty.

## 1. Mount Google Drive

In [ ]:
from google.colab import drive
drive.mount('/content/drive', force_remount=True)

## 2. Configuration

The Mip-NeRF 360 scenes wired into the experiments are `garden` and `bicycle`.
Set `MIP_SCENES` to any subset (or all nine) of:
`garden bicycle stump room counter kitchen bonsai flowers treehill`.

In [ ]:
import os
from pathlib import Path

HOME_DIR     = "/content/drive/MyDrive/Magister/Thesis/Script"
RAW_DIR      = Path(HOME_DIR) / "raw"        # Mip-NeRF 360 extraction + scratch (Drive)
DATASET_ROOT = Path(HOME_DIR) / "Dataset"    # prepared scenes (Drive)

MIP_SCENES = ["garden", "bicycle"]           # add "stump", "room", "counter", ... as needed

# --- Replica: deferred. Leave empty to skip; see §6 to enable. -------------- #
REPLICA_SCENES    = []
REPLICA_DEST      = DATASET_ROOT
REPLICA_LOCAL_SRC = str(DATASET_ROOT)
REPLICA_ZIP_URL   = ""
REPLICA_GDRIVE_ID = ""
REPLICA_HF_REPO   = ""

SCRATCH = Path("/content/_retrieve")         # fast local disk for download/merge
for p in (RAW_DIR, DATASET_ROOT, SCRATCH):
    p.mkdir(parents=True, exist_ok=True)
os.environ["HOME_DIR"]     = HOME_DIR
os.environ["RAW_DIR"]      = str(RAW_DIR)
os.environ["DATASET_ROOT"] = str(DATASET_ROOT)
print("raw dir      :", RAW_DIR)
print("dataset root :", DATASET_ROOT)
print("scenes       :", MIP_SCENES)

## 3. Download Mip-NeRF 360 (official `360_v2.zip`)

One several-GB archive; we extract only the requested scenes to `$RAW_DIR/<scene>`.
The zip already contains the COLMAP `sparse/0`, so these are trainable as-is (add
SAM2 labels later for the SGDC semantic arms).

In [ ]:
if MIP_SCENES:
    ZIP = SCRATCH / "360_v2.zip"
    if not ZIP.exists():
        !wget -q --show-progress -O "{ZIP}" https://storage.googleapis.com/gresearch/refraw360/360_v2.zip
    for s in MIP_SCENES:
        out = RAW_DIR / s
        if (out / "images").exists():
            print("skip (already have):", out)
            continue
        !unzip -q -n "{ZIP}" "{s}/*" -d "{RAW_DIR}"
        print(s, "->", out,
              "| images:", (out / "images").exists(),
              "| colmap:", (out / "sparse" / "0" / "cameras.bin").exists())
else:
    print("Mip-NeRF 360 retrieval skipped (MIP_SCENES is empty).")

## 4. Verify what landed on Drive

In [ ]:
from pathlib import Path


def _nfiles(d, pats=("*.png", "*.jpg", "*.jpeg")):
    d = Path(d)
    return sum(len(list(d.glob(p))) for p in pats)


for s in MIP_SCENES:
    d = RAW_DIR / s
    print(f"  {s:10s} images={_nfiles(d / 'images'):5d}  "
          f"colmap={(d / 'sparse' / '0' / 'cameras.bin').exists()}  -> {d}")

## 5. Next steps

These raw scenes are directly trainable. For the SGDC semantic arms they still
need labels -> `Prepare_Datasets.ipynb`:

- §5 HLOC SfM -> undistorted PINHOLE COLMAP (already provided by the zip, so this
  is only needed if you re-reconstruct),
- §6 SAM2 instance pseudo-labels -> `$DATASET_ROOT/mipnerf360/<scene>/semantic_instance`,
- §7 label/image alignment check,
- §9 the cross-dataset `garden_*` / `bicycle_*` runs (already wired in
  `experiments/runs.py`).

## 6. Optional: Replica (deferred)

Inactive while `REPLICA_SCENES == []`. To enable, set `REPLICA_SCENES` and one
source in §2. Ensures each scene sits at `REPLICA_DEST/<scene>/Sequence_1/`.

In [ ]:
import shutil, subprocess, sys
from pathlib import Path


def _unpack(archive: Path, dest: Path):
    """Extract a .zip or .tar(.gz) archive into dest."""
    import tarfile, zipfile
    archive, dest = Path(archive), Path(dest)
    dest.mkdir(parents=True, exist_ok=True)
    name = archive.name.lower()
    if name.endswith((".tar.gz", ".tgz", ".tar")):
        with tarfile.open(archive) as t:
            t.extractall(dest)
    else:
        with zipfile.ZipFile(archive) as z:
            z.extractall(dest)
    print("unpacked", archive.name, "->", dest)


def _gather_scenes(staging: Path, scenes):
    """Ensure <staging>/**/<scene> ends up at REPLICA_DEST/<scene>/Sequence_1."""
    staging = Path(staging)
    placed = []
    for s in scenes:
        dst = REPLICA_DEST / s
        if dst.exists():
            print("skip (already have):", dst)
            placed.append(s)
            continue
        hits = [p for p in staging.rglob(s) if p.is_dir()]
        if not hits:
            print("!! scene not found under", staging, ":", s)
            continue
        src = min(hits, key=lambda p: len(p.parts))   # shallowest match
        if src.resolve() == dst.resolve():
            print("already in place:", dst)
            placed.append(s)
            continue
        # preserve the <scene>/Sequence_* layout the trainer expects
        if (src / "images").is_dir() and not list(src.glob("Sequence_*")):
            wrap = dst / "Sequence_1"
            wrap.parent.mkdir(parents=True, exist_ok=True)
            shutil.copytree(src, wrap)
            print(f"placed {s}  <- {src}  (wrapped as Sequence_1)")
        else:
            shutil.copytree(src, dst)
            print(f"placed {s}  <- {src}")
        placed.append(s)
    return placed


if not REPLICA_SCENES:
    print("Replica retrieval skipped (REPLICA_SCENES is empty).")

elif REPLICA_LOCAL_SRC:
    _gather_scenes(Path(REPLICA_LOCAL_SRC), REPLICA_SCENES)

elif REPLICA_GDRIVE_ID:
    if not shutil.which("gdown"):
        subprocess.run([sys.executable, "-m", "pip", "install", "-q", "gdown"], check=True)
    import gdown
    staging = SCRATCH / "replica"
    staging.mkdir(parents=True, exist_ok=True)
    try:                                   # a folder link
        gdown.download_folder(id=REPLICA_GDRIVE_ID, output=str(staging),
                              quiet=False, use_cookies=False)
    except Exception as e:                 # a single-file link
        print("download_folder failed -> trying as a single file:", e)
        f = staging / "replica.zip"
        gdown.download(id=REPLICA_GDRIVE_ID, output=str(f), quiet=False)
        _unpack(f, staging)
    _gather_scenes(staging, REPLICA_SCENES)

elif REPLICA_HF_REPO:
    if not shutil.which("huggingface-cli"):
        subprocess.run([sys.executable, "-m", "pip", "install", "-q", "huggingface_hub"], check=True)
    from huggingface_hub import snapshot_download
    staging = SCRATCH / "replica"
    snapshot_download(
        repo_id=REPLICA_HF_REPO, repo_type="dataset", local_dir=str(staging),
        allow_patterns=[p for s in REPLICA_SCENES for p in (f"{s}/*", f"**/{s}/*")],
    )
    _gather_scenes(staging, REPLICA_SCENES)

elif REPLICA_ZIP_URL:
    staging = SCRATCH / "replica"
    staging.mkdir(parents=True, exist_ok=True)
    arc = staging / ("replica" + Path(REPLICA_ZIP_URL).suffix)
    !wget -q --show-progress --continue -O "{arc}" "{REPLICA_ZIP_URL}"
    _unpack(arc, staging)
    _gather_scenes(staging, REPLICA_SCENES)

else:
    print("No Replica source configured. Set one of REPLICA_LOCAL_SRC / REPLICA_ZIP_URL / "
          "REPLICA_GDRIVE_ID / REPLICA_HF_REPO in §2, or leave REPLICA_SCENES empty.")

## 7. (Optional) reclaim Colab scratch

The extracted scenes already live on Drive; the archive in `SCRATCH` can go.

In [ ]:
# import shutil
# shutil.rmtree(SCRATCH, ignore_errors=True)
# print("removed", SCRATCH)